# Model comparison

In [3]:
import json
import time
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from src.transformer_absa import TransformerABSA
from src.lexicon_absa import LexiconABSA
from src.llm_absa import LLMABSA
from collections import defaultdict

file_path = '../data/test_samples_small.json'
with open(file_path, 'r') as f:
    reviews_data = json.load(f)


grouped = defaultdict(list)
for item in reviews_data:
    text = item['text']
    for aspect_info in item.get('aspects', []):
        aspect = aspect_info.get('aspect')
        sentiment = aspect_info.get('sentiment')
        if aspect and sentiment:
            grouped[text].append((aspect, sentiment))

test_data = [(text, aspects) for text, aspects in grouped.items()]



# Helper functions
def extract_labels_from_absa_outputs(absa_outputs):
    label_dict = {}
    for aspect, sentiment in absa_outputs:
        label_dict[aspect.lower()] = sentiment.lower()
    return label_dict

def extract_labels_and_confidence_from_llm_outputs(llm_outputs):
    label_dict = {}
    conf_dict = {}
    for item in llm_outputs:
        aspect = item.aspect.lower()
        sentiment = item.sentiment.lower()
        confidence = item.confidence
        label_dict[aspect] = sentiment
        conf_dict[aspect] = confidence
    return label_dict, conf_dict

def evaluate_predictions(y_true, y_pred):
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)
    accuracy = accuracy_score(y_true, y_pred)
    return precision, recall, f1, accuracy

def average_confidence(conf_dict, aspects):
    confidences = [conf_dict.get(a.lower(), 0) for a in aspects]
    return sum(confidences) / len(confidences) if confidences else 0.0

def run_evaluation_on_model(model, test_data, is_llm=False):
    all_true, all_pred = [], []
    confidences = []
    total_time = 0.0
    
    for text, true_labels in test_data:
        start = time.time()
        pred_outputs = model.analyze(text)

        # --- Handle empty predictions safely ---
        if not pred_outputs:
            print(f"NO OUTPUT for text: {text[:80]}...")

            pred_dict = {}
            conf_dict = {}
        else:
            if is_llm:
                pred_dict, conf_dict = extract_labels_and_confidence_from_llm_outputs(pred_outputs)
            else:
                if isinstance(pred_outputs[0], tuple):
                    pred_dict = extract_labels_from_absa_outputs(pred_outputs)
                else:
                    try:
                        pred_tuples = [(obj.aspect, obj.sentiment) for obj in pred_outputs]
                        pred_dict = extract_labels_from_absa_outputs(pred_tuples)
                    except Exception as e:
                        print(f"Error converting prediction outputs: {e}")
                        raise
                conf_dict = {}

        elapsed = time.time() - start
        total_time += elapsed

        true_dict = extract_labels_from_absa_outputs(true_labels)
        
        for aspect in true_dict:
            all_true.append(true_dict[aspect])
            all_pred.append(pred_dict.get(aspect, 'missing'))
            if is_llm:
                confidences.append(conf_dict.get(aspect, 0.0))

    precision, recall, f1, accuracy = evaluate_predictions(all_true, all_pred)
    avg_conf = average_confidence(dict(zip([a.lower() for a in true_dict.keys()], confidences)), true_dict.keys()) if confidences else None
    
    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "accuracy": accuracy,
        "avg_confidence": avg_conf,
        "avg_inference_time": total_time / len(test_data) if len(test_data) > 0 else 0.0,
    }

lexicon_model = LexiconABSA()
transformer_model = TransformerABSA()
llm_model = LLMABSA()

# Run evaluation 
results = {}
results["Lexicon"] = run_evaluation_on_model(lexicon_model, test_data)
results["Transformer"] = run_evaluation_on_model(transformer_model, test_data)
results["LLM"] = run_evaluation_on_model(llm_model, test_data, is_llm=True)

for name, metrics in results.items():
    print(f"Results for {name}:")
    print(f" Precision: {metrics['precision']:.3f}")
    print(f" Recall: {metrics['recall']:.3f}")
    print(f" F1-score: {metrics['f1']:.3f}")
    print(f" Accuracy: {metrics['accuracy']:.3f}")
    if metrics['avg_confidence'] is not None:
        print(f" Avg Confidence: {metrics['avg_confidence']:.3f}")
    print(f" Avg Inference Time (s): {metrics['avg_inference_time']:.3f}\n")


C:\Users\khale\Desktop\KDG\year3\data6\data_6_llm_project\.venv\Lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
C:\Users\khale\Desktop\KDG\year3\data6\data_6_llm_project\.venv\Lib\site-packages\transformers\convert_slow_tokenizer.py:515: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


Results for Lexicon:
 Precision: 0.458
 Recall: 0.322
 F1-score: 0.335
 Accuracy: 0.322
 Avg Inference Time (s): 0.145

Results for Transformer:
 Precision: 0.505
 Recall: 0.305
 F1-score: 0.321
 Accuracy: 0.305
 Avg Inference Time (s): 0.389

Results for LLM:
 Precision: 0.708
 Recall: 0.475
 F1-score: 0.523
 Accuracy: 0.475
 Avg Confidence: 0.480
 Avg Inference Time (s): 12.734



The LexiconABSA, TransformerABSA, and LLMABSA models demonstrated varying performance on the test dataset for aspect-based sentiment analysis. The LLMABSA model showed the highest precision (0.708) and F1-score (0.523), indicating stronger ability to correctly identify and classify aspects, albeit with longer average inference time around 13 seconds. The LexiconABSA and TransformerABSA models performed moderately, with precision near 0.458 and 0.505, and F1-scores around 0.335 and 0.321 respectively, reflecting reasonable but lower accuracy in aspect sentiment detection. Their average inference times were significantly faster, approximately 0.15 seconds and 0.39 seconds respectively.

In summary, the lexicon-based approach offers computational efficiency and quick predictions suited for resource-constrained environments but with limited accuracy. The transformer-based model balances performance and inference speed but has room for optimization. The LLM-based approach delivers the highest accuracy at the cost of longer processing times, which may be acceptable for applications prioritizing precision over speed.